# Triagent: exploratory data analysis

A first look at Kubernetes issues, to check the modelling assumptions before building the pipeline.
Each section answers one question that affects a design decision.

Data: `data/raw/issues_sample.jsonl`, created by `scripts/fetch_sample.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw" / "issues_sample.jsonl"
REPORTS = ROOT / "reports" / "eda"
REPORTS.mkdir(parents=True, exist_ok=True)

TOKEN_LIMIT = 512  # DeBERTa's typical input length
TOKENS_PER_WORD = 1.3  # rough English average for subword tokenizers
PURPLE, TEAL, CORAL = "#7F77DD", "#1D9E75", "#D85A30"


def save(fig: plt.Figure, name: str) -> None:
    """Save a chart for the slides, then show it in the notebook."""
    fig.savefig(REPORTS / name, dpi=150, bbox_inches="tight")
    plt.show()

## Load the data

In [ ]:
df = pd.read_json(DATA, lines=True)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["body"] = df["body"].fillna("")
df["kind_labels"] = df["labels"].apply(lambda ls: [x for x in ls if x.startswith("kind/")])
df["sig_labels"] = df["labels"].apply(lambda ls: [x for x in ls if x.startswith("sig/")])
df["approx_tokens"] = (df["title"] + " " + df["body"]).str.split().str.len() * TOKENS_PER_WORD
df["has_code_block"] = df["body"].str.contains("```", regex=False)

first, last = df["created_at"].min(), df["created_at"].max()
print(f"{len(df)} issues, created {first:%d %b %Y} to {last:%d %b %Y}")
df[["number", "title", "state", "labels", "created_at"]].head()

## 1. Volume: how many issues arrive?

*Decision it informs: batch vs. streaming ingestion.*

In [ ]:
monthly = df.set_index("created_at").resample("MS").size()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(monthly.index.strftime("%b %y"), monthly.values, color=PURPLE)
ax.set(title="New issues per month", ylabel="Issues")
ax.tick_params(axis="x", rotation=45)
save(fig, "01_issues_per_month.png")
print(f"Average: {monthly.mean():.0f} issues per month")

## 2. Labels: are they balanced?

*Decisions it informs: which labels to predict, the metric, and imbalance handling.*

In [ ]:
def label_share(column: str, top: int = 12) -> pd.Series:
    """Percentage of issues carrying each label in `column`."""
    return df[column].explode().dropna().value_counts().head(top) / len(df) * 100


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, column, title in [
    (axes[0], "kind_labels", "Issue type (kind/*)"),
    (axes[1], "sig_labels", "Top 12 teams (sig/*)"),
]:
    share = label_share(column).sort_values()
    ax.barh(share.index, share.values, color=TEAL)
    ax.set(title=title, xlabel="% of issues")
save(fig, "02_label_distribution.png")

print(f"Issues with a kind/* label: {(df['kind_labels'].str.len() > 0).mean():.0%}")
print(f"Issues with a sig/* label:  {(df['sig_labels'].str.len() > 0).mean():.0%}")
print(f"Distinct sig/* labels:      {df['sig_labels'].explode().nunique()}")

## 3. Is it multi-label?

*Decision it informs: multi-label vs. single-label (multiclass) classification.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
for ax, column, title in [
    (axes[0], "kind_labels", "kind/* labels per issue"),
    (axes[1], "sig_labels", "sig/* labels per issue"),
]:
    counts = df[column].str.len().clip(upper=3).value_counts().sort_index()
    counts.index = [str(i) if i < 3 else "3+" for i in counts.index]
    ax.bar(counts.index, counts.values / len(df) * 100, color=PURPLE)
    ax.set(title=title, xlabel="Number of labels")
axes[0].set_ylabel("% of issues")
save(fig, "03_labels_per_issue.png")

print(f"Issues with more than one sig/* label: {(df['sig_labels'].str.len() > 1).mean():.0%}")

## 4. Text length and noise: does it fit the model?

*Decisions it informs: text cleaning, and DeBERTa (512 tokens) vs. ModernBERT (longer context).*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df["approx_tokens"].clip(upper=3000), bins=60, color=PURPLE)
ax.axvline(TOKEN_LIMIT, color=CORAL, linestyle="--", label=f"{TOKEN_LIMIT}-token limit")
ax.set(title="Issue length (title + body, approx. tokens)", xlabel="Tokens", ylabel="Issues")
ax.legend()
save(fig, "04_issue_length.png")

print(f"Longer than {TOKEN_LIMIT} tokens: {(df['approx_tokens'] > TOKEN_LIMIT).mean():.0%}")
print(f"Containing code blocks or logs: {df['has_code_block'].mean():.0%}")

## 5. Duplicates and triage status

*Decisions it informs: the duplicate-retrieval evaluation set, and delayed-label monitoring.*

In [ ]:
def has_label(name: str) -> pd.Series:
    return df["labels"].apply(lambda labels: name in labels)


duplicates = has_label("triage/duplicate") | (df["state_reason"] == "duplicate")
print(f"Marked as duplicate:        {duplicates.sum()} issues ({duplicates.mean():.1%})")
print(f"Still labelled needs-triage: {has_label('needs-triage').mean():.0%}")
print(f"Labelled triage/accepted:    {has_label('triage/accepted').mean():.0%}")
print()
df["state_reason"].fillna("still open").value_counts(normalize=True).mul(100).round(1)

## Key findings

Fill in with the numbers from your run.

1. **Label coverage:** …% of issues have a `kind/*` label and …% a `sig/*` label, enough to learn from without manual labelling.
2. **Imbalance:** `sig/node` alone covers …%, with a long tail of SIGs. → Predict the top ~10 SIGs plus "other", use macro F1, weight the loss, tune a threshold per label.
3. **Multi-label:** …% of issues have more than one SIG. → Multi-label classification.
4. **Long, noisy text:** …% exceed 512 tokens and …% contain code or logs. → Strip code blocks before truncating; compare DeBERTa with ModernBERT.
5. **Scarce duplicates:** only … duplicates in the sample. → Ingest several years of history and mine "duplicate of #N" comments for the evaluation set.
6. **Triage lags labelling:** …% still `needs-triage`. → Train on settled labels; monitor accuracy as labels arrive later.
7. **Volume:** about … issues per month. → Daily batch ingestion is enough.